# Media-plan optimizer: production-backed demo

This notebook exercises the repository's public optimization service against the bundled deterministic synthetic catalog. It does not reproduce planner logic: all scenarios call `optimize_media_plan`, and response curves use the production curve implementation.

## 1. Imports and setup

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "mediaplan_optimizer").is_dir():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "mediaplan_optimizer").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repository root or notebooks directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px

from mediaplan_optimizer import (
    OptimizationRequest,
    generate_catalog,
    load_catalog,
    optimize_media_plan,
)
from mediaplan_optimizer.curves import channel_response

pd.options.display.float_format = "{:,.2f}".format
catalog = load_catalog()
print(f"Repository discovered successfully; loaded {len(catalog)} channels.")

Repository discovered successfully; loaded 8 channels.


## 2. Synthetic channel catalog

The checked-in catalog is deterministic and illustrative. Monetary columns are Russian rubles; capacity is daily delivered impressions or contacts.

In [2]:
catalog_df = pd.DataFrame([channel.model_dump() for channel in catalog.values()])
catalog_df[[
    "name", "channel_type", "daily_capacity", "daily_reach_capacity",
    "response_scale_rub", "max_daily_spend", "base_ctr", "base_cr"
]]

,name,channel_type,daily_capacity,daily_reach_capacity,response_scale_rub,max_daily_spend,base_ctr,base_cr
0,Social Network 1,media,"2,218,536.00","1,101,018.00","496,446.00","1,290,644.00",0.02,0.04
1,Social Network 2,media,"1,836,692.00","1,047,201.00","492,790.00","1,203,807.00",0.01,0.02
2,Social Network 3,media,"1,291,540.00","639,111.00","233,691.00","903,705.00",0.01,0.03
3,Programmatic,media,"4,490,484.00","1,582,855.00","669,708.00","2,012,401.00",0.00,0.02
4,Marketplace 1,media,"912,863.00","589,528.00","424,156.00","1,204,451.00",0.02,0.05
5,Marketplace 2,media,"861,009.00","539,473.00","373,211.00","752,768.00",0.03,0.04
6,Marketplace 3,media,"529,239.00","317,641.00","187,390.00","689,619.00",0.02,0.03
7,SMS,sms,"396,112.00","382,150.00","139,331.00","347,691.00",0.03,0.03


## 3. Production response curves

Each line below is evaluated by `mediaplan_optimizer.curves.channel_response`. The flattening lines expose the modeled diminishing returns that drive allocation.

In [3]:
curve_rows = []
for name, channel in catalog.items():
    for spend in np.linspace(0, channel.max_daily_spend, 40):
        response = channel_response(spend, channel)
        curve_rows.append({
            "channel": name,
            "daily_spend_rub": spend,
            "daily_conversions": response.conversions,
        })

curves_df = pd.DataFrame(curve_rows)
curve_figure = px.line(
    curves_df,
    x="daily_spend_rub",
    y="daily_conversions",
    color="channel",
    title="Daily conversion response by channel",
    labels={"daily_spend_rub": "Daily spend, RUB", "daily_conversions": "Conversions"},
)
curve_figure.update_layout(legend_title_text="Channel", hovermode="x unified")
curve_figure.show(config={"displayModeBar": False})

## 4. Type A: maximize conversions with RUB 1.2M over 21 days

Type A treats budget as a cap and maximizes the requested KPI across the full catalog.

In [4]:
type_a_request = OptimizationRequest(
    task_type="A",
    horizon_days=21,
    objective_metric="conversions",
    budget=1_200_000,
)
type_a = optimize_media_plan(type_a_request, catalog)
assert type_a.summary is not None

type_a_summary = pd.DataFrame([type_a.summary.model_dump()])
print(f"Status: {type_a.status.value}; solver validation: {type_a.optimizer_metadata.validation_passed}")
type_a_summary[[
    "spend", "available_budget", "unspent_budget", "conversions",
    "clicks", "reach", "cpa"
]]

Status: optimal; solver validation: True


,spend,available_budget,unspent_budget,conversions,clicks,reach,cpa
0,"1,200,000.00","1,200,000.00",0.00,"3,069.98","92,703.25","2,491,290.97",390.88


In [5]:
type_a_channels = pd.DataFrame([row.model_dump() for row in type_a.channel_metrics])
type_a_channels[["channel", "spend", "conversions", "clicks", "cpa"]].sort_values(
    "spend", ascending=False
).reset_index(drop=True)

,channel,spend,conversions,clicks,cpa
0,Social Network 1,"666,639.55","1,681.15","45,084.87",396.54
1,SMS,"434,272.87","1,146.44","39,569.40",378.80
2,Social Network 3,"99,087.58",242.40,"8,048.98",408.78
3,Social Network 2,0.00,0.00,0.00,NaN
4,Programmatic,0.00,0.00,0.00,NaN
5,Marketplace 1,0.00,0.00,0.00,NaN
6,Marketplace 2,0.00,0.00,0.00,NaN
7,Marketplace 3,0.00,0.00,0.00,NaN


## 5. Feasible Type B: reach 10,000 clicks in 14 days

Type B first checks catalog capacity, then minimizes spend while meeting the KPI target.

In [6]:
type_b_request = OptimizationRequest(
    task_type="B",
    horizon_days=14,
    objective_metric="clicks",
    target_value=10_000,
)
type_b = optimize_media_plan(type_b_request, catalog)
assert type_b.summary is not None and type_b.feasibility is not None

print(type_b.feasibility.explanation)
pd.DataFrame([{
    "status": type_b.status.value,
    "target": type_b.summary.requested_target,
    "achieved": type_b.summary.achieved_target,
    "minimum_spend_rub": type_b.summary.spend,
    "cpa": type_b.summary.cpa,
    "14_day_capacity": type_b.feasibility.maximum_achievable,
}])

The selected channels can achieve the requested KPI within the horizon.


,status,target,achieved,minimum_spend_rub,cpa,14_day_capacity
0,optimal,"10,000.00","10,000.01","104,138.52",358.17,"1,862,249.97"


## 6. Infeasible Type B: target exceeds 14-day capacity

The public service returns a structured infeasibility result without running the optimizer.

In [7]:
infeasible_request = OptimizationRequest(
    task_type="B",
    horizon_days=14,
    objective_metric="conversions",
    target_value=1_000_000,
)
infeasible = optimize_media_plan(infeasible_request, catalog)
assert infeasible.summary is None and infeasible.feasibility is not None

pd.DataFrame([{
    "status": infeasible.status.value,
    "reason": infeasible.feasibility.reason.value,
    "requested": infeasible.feasibility.requested_value,
    "maximum_achievable": infeasible.feasibility.maximum_achievable,
    "target_gap": infeasible.feasibility.target_gap,
    "minimum_feasible_horizon": infeasible.feasibility.minimum_feasible_horizon,
    "recommendation": "; ".join(infeasible.feasibility.recommendations),
}])

,status,reason,requested,maximum_achievable,target_gap,minimum_feasible_horizon,recommendation
0,infeasible,HORIZON_TOO_SHORT,"1,000,000.00","55,334.68","944,665.32",254,Increase the horizon to approximately 254 days...


## 7. Plan plots

The first plot compares optimized channel spend. The second reconstructs the feasible Type B cumulative trajectory from returned daily allocations.

In [8]:
plot_rows = []
for scenario, result in [("Type A", type_a), ("Type B", type_b)]:
    plot_rows.extend(
        {"scenario": scenario, "channel": row.channel, "spend_rub": row.spend}
        for row in result.channel_metrics
    )
spend_plot_df = pd.DataFrame(plot_rows)
spend_figure = px.bar(
    spend_plot_df,
    x="channel",
    y="spend_rub",
    color="scenario",
    barmode="group",
    title="Optimized spend by channel",
    labels={"spend_rub": "Spend, RUB", "channel": "Channel"},
)
spend_figure.update_layout(xaxis_tickangle=-30)
spend_figure.show(config={"displayModeBar": False})

daily_type_b = (
    pd.DataFrame([row.model_dump() for row in type_b.allocations])
    .groupby("day", as_index=False)["clicks"]
    .sum()
)
daily_type_b["cumulative_clicks"] = daily_type_b["clicks"].cumsum()
trajectory_figure = px.line(
    daily_type_b,
    x="day",
    y="cumulative_clicks",
    markers=True,
    title="Feasible Type B cumulative clicks",
    labels={"day": "Day", "cumulative_clicks": "Cumulative clicks"},
)
trajectory_figure.add_hline(y=type_b_request.target_value, line_dash="dash", annotation_text="Target")
trajectory_figure.show(config={"displayModeBar": False})

## 8. Reproducibility checks

Repeated seed-42 generation, repeated loading of the bundled catalog, and repeated optimization must each produce stable results.

In [9]:
generated_once = generate_catalog(seed=42)
generated_twice = generate_catalog(seed=42)
generated_catalog_is_reproducible = (
    {name: item.model_dump() for name, item in generated_once.items()}
    == {name: item.model_dump() for name, item in generated_twice.items()}
)
reloaded_catalog = load_catalog()
bundled_catalog_is_reproducible = (
    {name: item.model_dump() for name, item in reloaded_catalog.items()}
    == {name: item.model_dump() for name, item in catalog.items()}
)
repeated_type_a = optimize_media_plan(type_a_request, catalog)
assert repeated_type_a.summary is not None
plan_is_reproducible = np.isclose(
    repeated_type_a.summary.achieved_target, type_a.summary.achieved_target, rtol=1e-10
) and np.isclose(repeated_type_a.summary.spend, type_a.summary.spend, rtol=1e-10)
assert generated_catalog_is_reproducible and bundled_catalog_is_reproducible and plan_is_reproducible
pd.DataFrame([{
    "catalog_seed": 42,
    "generated_catalog_reproduced": generated_catalog_is_reproducible,
    "bundled_catalog_reloaded": bundled_catalog_is_reproducible,
    "type_a_plan_reproduced": plan_is_reproducible,
}])

,catalog_seed,generated_catalog_reproduced,bundled_catalog_reloaded,type_a_plan_reproduced
0,42,True,True,True


## 9. Conclusions

- The public service turns a fixed budget into a validated KPI-maximizing plan and returns spend, delivery, and efficiency metrics.
- A feasible KPI target produces a minimum-spend plan after an explicit capacity check.
- An impossible target produces a structured reason, quantified gap, and horizon recommendation instead of a misleading allocation.
- Catalog generation and repeated optimization are deterministic for the demonstrated inputs.

All figures and scenario values above come from production modules and the checked-in synthetic assumptions.